In [ ]:
# %% Deep learning - Section 21.200
#    Pretraining with autoencoders

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% FMNIST data

# Transfors
transform = T.Compose([ T.ToTensor(),
                        T.RandomHorizontalFlip(p=.5),
                        T.Normalize(.5,.5) ])

# Import data and apply the transform
train_set = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
dev_test  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Randomly split into proper dev_set and test_set
rand_idx = np.random.permutation(10000)
dev_set  = Subset(dev_test,rand_idx[:6000])
test_set = Subset(dev_test,rand_idx[6000:])

# Convert into DataLoader objects
batchsize    = 32
train_loader = DataLoader(train_set,batch_size=batchsize,shuffle=True,drop_last=True)
dev_loader   = DataLoader(dev_set,batch_size=len(dev_set))
test_loader  = DataLoader(test_set,batch_size=len(test_set))


In [ ]:
# %% inspect a few random images

X,y = next(iter(test_loader))

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,6,figsize=(phi*6,6))

for (i,ax) in enumerate(axs.flatten()):

    # Get image and undo normalisation
    pic = torch.squeeze(X.data[i])
    pic = pic/2 + .5

    # Label
    label = train_set.classes[y[i]]

    ax.imshow(pic,cmap='gray')
    ax.text(14,0,label,ha='center',fontweight='bold',color='k',backgroundcolor='y')
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure21_pretrain_with_autoencoders.png')
plt.show()
files.download('figure21_pretrain_with_autoencoders.png')


In [27]:
# %% Function to generate the autencoder model

def gen_ae_model(printtoggle=False):

    class AE_net(nn.Module):
        def __init__(self,printtoggle):
            super().__init__()

            # Print toggle
            self.print = printtoggle

            # Encoder layers
            self.encconv1  = nn.Conv2d( 1,16,3,padding=1,stride=2) # output size: (28+2*1-3)/2 + 1 = 14
            self.encconv2  = nn.Conv2d(16,32,3,padding=1,stride=2) # output size: (14+2*1-3)/2 + 1 = 7

            # Decoder layers
            self.decconv1  = nn.ConvTranspose2d(32,16,4,padding=1,stride=2) # output size: (28+2*1-3)/2 + 1 = 14
            self.decconv2  = nn.ConvTranspose2d(16,1 ,4,padding=1,stride=2) # output size: (14+2*1-3)/2 + 1 = 7

        def forward(self,x):

            if self.print: print(f'Input: {list(x.shape)}')

            # First encoder layer
            x = F.leaky_relu(self.encconv1(x))
            if self.print: print(f'First encoder layer: {list(x.shape)}')

            # Second encoder layer
            conv2_act = F.leaky_relu(self.encconv2(x))
            if self.print: print(f'Second encoder layer: {list(conv2_act.shape)}')

            # First decoder layer
            x = F.leaky_relu(self.decconv1(conv2_act))
            if self.print: print(f'First decoder layer: {list(x.shape)}')

            # Second decoder layer
            x = F.leaky_relu(self.decconv2(x))
            if self.print: print(f'Second decoder layer: {list(x.shape)}')

            return x,conv2_act

    # Create model instance
    net = AE_net(printtoggle)

    # Loss function
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(net.parameters(),lr=.001)

    return net,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

tmp_net,loss_fun,optimizer = gen_ae_model(True)

X,y         = next(iter(train_loader))
yHat,latent = tmp_net(X)

# Check sizes of output
print('\nOutput size:')
print(yHat.shape)

# Check latent
print('\nLatent size:')
print(latent.shape)

# Check loss
loss = loss_fun(yHat,X)
print(' ')
print('Loss:')
print(loss)


In [34]:
# %% Function to train the autoencoder model

def train_ae_model():

    # Parameters, model instance, inizialise vars
    numepochs = 10
    ae_net,loss_fun,optimizer = gen_ae_model()

    # Ship to GPU
    ae_net.to(device)

    # Preallocate losses
    train_loss = torch.zeros(numepochs)
    dev_loss   = torch.zeros(numepochs)

    # Loop over epochs
    for epochi in range(numepochs):

        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat,_ = ae_net(X)
            loss   = loss_fun(yHat,X)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss from this batch
            batch_loss.append(loss.item())

        # Average losses across batches
        train_loss[epochi] = np.mean(batch_loss)

        # Dev performance
        ae_net.eval()
        X,y = next(iter(dev_loader))

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        with torch.no_grad():
            yHat,_ = ae_net(X)
            loss   = loss_fun(yHat,X)

        # Dev loss (one batch)
        dev_loss[epochi] = loss.item()
        ae_net.train()

    return train_loss,dev_loss,ae_net


In [35]:
# %% Fit the AE model

# Takes ~3 mins on GPU
train_loss,dev_loss,ae_net = train_ae_model()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(train_loss,'s-',label='Train')
plt.plot(dev_loss,'o-',label='Dev')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Model loss')
plt.legend()

plt.savefig('figure22_pretrain_with_autoencoders.png')
plt.show()
files.download('figure22_pretrain_with_autoencoders.png')


In [ ]:
# %% Plotting

# Get some data
X,y = next(iter(dev_loader))

# Forward propagation and loss
ae_net.cpu()
ae_net.eval()
yHat,_ = ae_net(X)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,10,figsize=(2*phi*5,5))

for i in range(10):

    # Predicted (undo normalisation)
    pic = yHat[i,0,:,:].detach()
    pic = pic/2 + .5
    axs[0,i].imshow(pic,cmap='gray')
    axs[0,i].axis('off')

    # Actual (undo normalisation)
    pic = X[i,0,:,:].detach()
    pic = pic/2 + .5
    axs[1,i].imshow(pic,cmap='gray')
    axs[1,i].axis('off')

    if i==0:
        axs[0,0].text(-6,14,'Reconstructed',rotation=90,va='center')
        axs[1,0].text(-6,14,'Original',rotation=90,va='center')

plt.savefig('figure23_pretrain_with_autoencoders.png')
plt.show()
files.download('figure23_pretrain_with_autoencoders.png')


In [39]:
# %% Function to generate the classifier model

def gen_classifier_model(printtoggle=False):

    class CNN_net(nn.Module):
        def __init__(self,printtoggle):
            super().__init__()

            # Print toggle
            self.print = printtoggle

            # Encoder layers
            self.encconv1  = nn.Conv2d( 1,16,3,padding=1,stride=2) # output size: (28+2*1-3)/2 + 1 = 14
            self.encconv2  = nn.Conv2d(16,32,3,padding=1,stride=2) # output size: (14+2*1-3)/2 + 1 = 7

            # Linear layers
            self.fc1  = nn.Linear(7*7*32,50)
            self.fc2  = nn.Linear(50,10)

        def forward(self,x):

            if self.print: print(f'Input: {list(x.shape)}')

            # First encoder layer
            x = F.leaky_relu(self.encconv1(x))
            if self.print: print(f'First encoder layer: {list(x.shape)}')

            # Second encoder layer
            conv2_act = F.leaky_relu(self.encconv2(x))
            if self.print: print(f'Second encoder layer: {list(conv2_act .shape)}')

            # Vectorise
            n_units = conv2_act.shape.numel()/conv2_act.shape[0]
            x       = conv2_act.view(-1,int(n_units))
            if self.print: print(f'Post-convolution vectorised: {list(x.shape)}')

            # Linear layers
            x = F.leaky_relu(self.fc1(x))
            if self.print: print(f'First linear layer: {list(x.shape)}')

            x = self.fc2(x)
            if self.print: print(f'Second linear layer: {list(x.shape)}')

            return x,conv2_act

    # Create model instance
    net = CNN_net(printtoggle)

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(net.parameters(),lr=.001)

    return net,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

tmp_net,loss_fun,optimizer = gen_classifier_model(True)

X,y         = next(iter(train_loader))
yHat,latent = tmp_net(X)

# Check sizes of output
print('\nOutput size:')
print(yHat.shape)

# Check latent shape
print('\nLatent size:')
print(latent.shape)

# Check loss
loss = loss_fun(yHat,y)
print()
print('Loss:')
print(loss)


In [41]:
# %% Function to train the classifier model

def train_classifier_model(net,loss_fun,optimizer):

    # Parameters
    numepochs = 10

    # Ship to GPU
    net.to(device)

    # Preallocate losses and accuracies
    train_loss = torch.zeros(numepochs)
    dev_loss   = torch.zeros(numepochs)
    train_acc  = torch.zeros(numepochs)
    dev_acc    = torch.zeros(numepochs)

    # Loop over epochs
    for epochi in range(numepochs):

        batch_loss = []
        batch_acc  = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat,_ = net(X)
            loss   = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            batch_loss.append(loss.item())
            batch_acc.append(torch.mean((torch.argmax(yHat,axis=1) == y).float()).item())

        # Average losses across batches
        train_loss[epochi] = np.mean(batch_loss)
        train_acc[epochi]  = 100*np.mean(batch_acc)

        # Dev performance
        net.eval()
        X,y = next(iter(dev_loader))

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        with torch.no_grad():
            yHat,_ = net(X)
            loss   = loss_fun(yHat,y)

        # Dev loss (one batch)
        dev_loss[epochi] = loss.item()
        dev_acc[epochi]  = 100 * torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()

        net.train()

    return train_acc,dev_acc,train_loss,dev_loss,net


In [44]:
# %% Train a new model from scratch

# Create a 'naive' classifier network (trained from scratch), takes ~3 mins
naive_net,loss_fun,optimizer = gen_classifier_model()
train_acc_naive,dev_acc_naive,train_loss_naive,dev_loss_naive,naive_net = train_classifier_model(naive_net,loss_fun,optimizer)

# Evaluate on test set (push data to GPU)
naive_net.eval()
X,y = next(iter(test_loader))

X = X.to(device)
y = y.to(device)

# Forward propagation and loss
with torch.no_grad():
    yHat,_ = naive_net(X)
    loss   = loss_fun(yHat,y)

# Loss and accuracy from test batch
test_loss_naive = loss.item()
test_acc_naive  = 100 * torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_naive,'s-',label='Train')
ax[0].plot(dev_loss_naive,'o-',label='Dev')
ax[0].plot(len(dev_loss_naive)-1,test_loss_naive,'r*',markersize=15,label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')

ax[1].plot(train_acc_naive,'s-',label='Train')
ax[1].plot(dev_acc_naive,'o-',label='Dev')
ax[1].plot(len(dev_acc_naive)-1,test_acc_naive,'r*',markersize=15,label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model [dev,test] accuracy: [{dev_acc_naive[-1]:.2f}%,{test_acc_naive:.2f}%]')
ax[1].legend()

plt.savefig('figure24_pretrain_with_autoencoders.png')
plt.show()
files.download('figure24_pretrain_with_autoencoders.png')


In [ ]:
# %% Build pretrained model

pretrained_net,loss_fun,optimizer = gen_classifier_model()

# Note about the code below :
# Both networks have the same number of layers overall; in other applications
# you may need to modify the code to find the matching layers

# Replace the conv. weights in target model with encoder weights from source model
for target,source in zip(pretrained_net.named_parameters(),ae_net.named_parameters()):
    print('Pretrained: ' + target[0] + '  --  AE_net: ' + source[0])
    if 'enc' in target[0]:
        target[1].data = copy.deepcopy( source[1].data )


In [48]:
# %% Fine-tune the pretrained model

# Tuning, takes ~3 mins
train_acc_pre,dev_acc_pre,train_loss_pre,dev_loss_pre,pretrained_net = train_classifier_model(pretrained_net,loss_fun,optimizer)

# Evaluate on test set
pretrained_net.eval()
X,y = next(iter(test_loader))

X = X.to(device)
y = y.to(device)

with torch.no_grad():
    yHat,_ = pretrained_net(X)
    loss   = loss_fun(yHat,y)

# Loss and accuracy from test batch
test_loss_pre = loss.item()
test_acc_pre  = 100*torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_pre,'s-',color='tab:red',label='Train (pretrained)')
ax[0].plot(dev_loss_pre,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[0].plot(len(dev_loss_pre)-1,test_loss_pre,'*',color='tab:red',markersize=10,label='Test (pretrained)')
ax[0].plot(train_loss_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[0].plot(dev_loss_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[0].plot(len(dev_loss_naive)-1,test_loss_naive,'*',color='tab:blue',markersize=10,label='Test (naïve)')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc_pre,'s-',color='tab:red',label='Train (pretrained)')
ax[1].plot(dev_acc_pre,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[1].plot(len(dev_acc_pre)-1,test_acc_pre,'*',color='tab:red',markersize=10,label='Test (pretrained)')
ax[1].plot(train_acc_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[1].plot(dev_acc_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[1].plot(len(dev_acc_naive)-1,test_acc_naive,'*',color='tab:blue',markersize=10,label='Test (naïve)')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final tesst [naïve, pretrained] accuracy: [{test_acc_naive:.2f}%, {test_acc_pre:.2f}%]')
ax[1].legend()

plt.savefig('figure25_pretrain_with_autoencoders.png')
plt.show()
files.download('figure25_pretrain_with_autoencoders.png')


In [ ]:
# %% Plotting

# Grab one image
x = X[20,:,:,:].view(1,1,28,28)

# Compute the activations of the first layer (exclude the bias, it is simply a constant)
layer1_act_pre = F.relu( F.conv2d(x,pretrained_net.encconv1.weight) )
layer1_act_nai = F.relu( F.conv2d(x,naive_net.encconv1.weight) )

# Show the feature map activations for the pretrained model
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,8,figsize=(2*phi*5,5))

for i,ax in enumerate(axs.flatten()):
    act = torch.squeeze(layer1_act_pre[0,i,:,:]).detach().cpu()
    ax.imshow(act,cmap='gray')
    ax.axis('off')

plt.suptitle('Pretrained activations',y=.9)

plt.savefig('figure26_pretrain_with_autoencoders.png')
plt.show()
files.download('figure26_pretrain_with_autoencoders.png')

# Show the feature map activations for the naive model
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,8,figsize=(2*phi*5,5))
for i,ax in enumerate(axs.flatten()):
    act = torch.squeeze(layer1_act_nai[0,i,:,:]).detach().cpu()
    ax.imshow(act,cmap='gray')
    ax.axis('off')

plt.suptitle('Naive activations',y=.9)

plt.savefig('figure27_pretrain_with_autoencoders.png')
plt.show()
files.download('figure27_pretrain_with_autoencoders.png')


In [ ]:
# %% Exercise 1
#    Computing the activation of the first layer is fairly straight forward, but gets more complicated as you go deeper
#    into the model. Perhaps more insight into the differences between the native and pretrained models will come from
#    inspecting the activation of the second convolution layer. To examine this, you can modify the network classes to
#    output the activation maps (see lecture "Examine feature map activations" from CNN section). Does examining the
#    feature map activations of this layer provide any more insight into the differences in performance between the
#    two different models?

# Well... that's messy

# Grab second layer activations and plot the 32 maps
x = X[20,:,:,:].view(1,1,28,28).to(device)

_, layer2_act_pre = pretrained_net(x)
_, layer2_act_nai = naive_net(x)

layer2_act_pre = layer2_act_pre.detach().cpu()
layer2_act_nai = layer2_act_nai.detach().cpu()

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,8,figsize=(2*phi*10,10))

for i,ax in enumerate(axs.flatten()):
    act = torch.squeeze(layer2_act_pre[0,i,:,:])
    ax.imshow(act,cmap='gray')
    ax.axis('off')

plt.suptitle('Pretrained activations (layer 2)',y=.9)

plt.savefig('figure28_pretrain_with_autoencoders_extra1.png')
plt.show()
files.download('figure28_pretrain_with_autoencoders_extra1.png')

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,8,figsize=(2*phi*10,10))

for i,ax in enumerate(axs.flatten()):
    act = torch.squeeze(layer2_act_nai[0,i,:,:])
    ax.imshow(act,cmap='gray')
    ax.axis('off')

plt.suptitle('Naive activations (layer 2)',y=.9)

plt.savefig('figure29_pretrain_with_autoencoders_extra1.png')
plt.show()
files.download('figure29_pretrain_with_autoencoders_extra1.png')


In [54]:
# %% Exercise 2
#    Here we used pretraining as a "smart weight initialization," but you can also pretrain and freeze the copied layers.
#    Given the performance seen above, it's unlikely that freezing the convolution layers (instead of allowing them to
#    continue learning) will increase performance, but it (1) is a great opportunity for you to practice freezing layers
#    and (2) will be interesting to see whether the frozen model does worse or the same as the pretrainNet.

# Lower performance as expected

# Copy and freeze the convolutional layers, takes ~3 mins
pretrained_net_f,loss_fun,_ = gen_classifier_model()

pretrained_net_f.encconv1.weight.data = ae_net.encconv1.weight.data.clone()
pretrained_net_f.encconv1.bias.data   = ae_net.encconv1.bias.data.clone()
pretrained_net_f.encconv2.weight.data = ae_net.encconv2.weight.data.clone()
pretrained_net_f.encconv2.bias.data   = ae_net.encconv2.bias.data.clone()

for param in pretrained_net_f.encconv1.parameters():
    param.requires_grad = False

for param in pretrained_net_f.encconv2.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, pretrained_net_f.parameters()),lr=0.001)

train_acc_pre_f,dev_acc_pre_f,train_loss_pre_f,dev_loss_pre_f,pretrained_net_f = train_classifier_model(pretrained_net_f,loss_fun,optimizer)

pretrained_net_f.eval()
X,y = next(iter(test_loader))

X = X.to(device)
y = y.to(device)

with torch.no_grad():
    yHat,_ = pretrained_net_f(X)
    loss   = loss_fun(yHat,y)

test_loss_pre_f = loss.item()
test_acc_pre_f  = 100*torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()


In [ ]:
# %% Exercise 2
#    Continue ...

# Plotting
phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_pre_f,'s-',color='tab:red',label='Train (pretrained)')
ax[0].plot(dev_loss_pre_f,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[0].plot(len(dev_loss_pre_f)-1,test_loss_pre_f,'*',color='tab:red',markersize=10,label='Test (pretrained)')
ax[0].plot(train_loss_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[0].plot(dev_loss_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[0].plot(len(dev_loss_naive)-1,test_loss_naive,'*',color='tab:blue',markersize=10,label='Test (naïve)')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc_pre_f,'s-',color='tab:red',label='Train (pretrained)')
ax[1].plot(dev_acc_pre_f,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[1].plot(len(dev_acc_pre_f)-1,test_acc_pre,'*',color='tab:red',markersize=10,label='Test (pretrained)')
ax[1].plot(train_acc_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[1].plot(dev_acc_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[1].plot(len(dev_acc_naive)-1,test_acc_naive,'*',color='tab:blue',markersize=10,label='Test (naïve)')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final test [naïve, pretrained-freezed] accuracy: [{test_acc_naive:.2f}%, {test_acc_pre_f:.2f}%]')
ax[1].legend()

plt.savefig('figure30_pretrain_with_autoencoders_extra2.png')
plt.show()
files.download('figure30_pretrain_with_autoencoders_extra2.png')


In [57]:
# %% Self-inflicted exercise 3
#    Compare a naïve and a pretrained model built also incorporating the first
#    decoding layer of the autoencoder, to see if the performance of the
#    pretrained model improves

#

# Model
def gen_classifier_model_extra(printtoggle=False):

    class CNN_net(nn.Module):
        def __init__(self,printtoggle):
            super().__init__()

            self.print = printtoggle

            self.encconv1  = nn.Conv2d( 1,16,3,padding=1,stride=2)
            self.encconv2  = nn.Conv2d(16,32,3,padding=1,stride=2)
            self.decconv1  = nn.ConvTranspose2d(32,16,4,padding=1,stride=2)

            self.fc1  = nn.Linear(14*14*16,50)
            self.fc2  = nn.Linear(50,10)

        def forward(self,x):

            if self.print: print(f'Input: {list(x.shape)}')

            x = F.leaky_relu(self.encconv1(x))
            if self.print: print(f'First encoder layer: {list(x.shape)}')

            conv2_act = F.leaky_relu(self.encconv2(x))
            if self.print: print(f'Second encoder layer: {list(conv2_act.shape)}')

            x = F.leaky_relu(self.decconv1(conv2_act))
            if self.print: print(f'First decoder layer: {list(x.shape)}')

            n_units = x.shape.numel()/x.shape[0]
            x       = x.view(-1,int(n_units))
            if self.print: print(f'Post-convolution vectorised: {list(x.shape)}')

            x = F.leaky_relu(self.fc1(x))
            if self.print: print(f'First linear layer: {list(x.shape)}')

            x = self.fc2(x)
            if self.print: print(f'Second linear layer: {list(x.shape)}')

            return x,conv2_act

    net = CNN_net(printtoggle)
    loss_fun = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(),lr=.001)

    return net,loss_fun,optimizer


In [58]:
# %% Self-inflicted exercise 3
#    Continue ...

# Train a new model from scratch
naive_net,loss_fun,optimizer = gen_classifier_model_extra()
train_acc_naive,dev_acc_naive,train_loss_naive,dev_loss_naive,naive_net = train_classifier_model(naive_net,loss_fun,optimizer)

naive_net.eval()
X,y = next(iter(test_loader))

X = X.to(device)
y = y.to(device)

with torch.no_grad():
    yHat,_ = naive_net(X)
    loss   = loss_fun(yHat,y)

test_loss_naive = loss.item()
test_acc_naive  = 100 * torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()


In [ ]:
# %% Self-inflicted exercise 3
#    Continue ...

# Plotting
phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_naive,'s-',label='Train')
ax[0].plot(dev_loss_naive,'o-',label='Dev')
ax[0].plot(len(dev_loss_naive)-1,test_loss_naive,'r*',markersize=15,label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')

ax[1].plot(train_acc_naive,'s-',label='Train')
ax[1].plot(dev_acc_naive,'o-',label='Dev')
ax[1].plot(len(dev_acc_naive)-1,test_acc_naive,'r*',markersize=15,label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model [dev,test] accuracy: [{dev_acc_naive[-1]:.2f}%,{test_acc_naive:.2f}%]')
ax[1].legend()

plt.savefig('figure31_pretrain_with_autoencoders_extra3.png')
plt.show()
files.download('figure31_pretrain_with_autoencoders_extra3.png')


In [60]:
# %% Self-inflicted exercise 3
#    Continue ...

# Copy and freeze the convolutional layers, takes ~3 mins
pretrained_net_e,loss_fun,optimizer = gen_classifier_model()

pretrained_net_e.encconv1.weight.data = ae_net.encconv1.weight.data.clone()
pretrained_net_e.encconv1.bias.data   = ae_net.encconv1.bias.data.clone()
pretrained_net_e.encconv2.weight.data = ae_net.encconv2.weight.data.clone()
pretrained_net_e.encconv2.bias.data   = ae_net.encconv2.bias.data.clone()
pretrained_net_e.decconv1.weight.data = ae_net.decconv1.weight.data.clone()
pretrained_net_e.decconv1.bias.data   = ae_net.decconv1.bias.data.clone()

train_acc_pre_e,dev_acc_pre_e,train_loss_pre_e,dev_loss_pre_e,pretrained_net_e = train_classifier_model(pretrained_net_e,loss_fun,optimizer)

pretrained_net_e.eval()
X,y = next(iter(test_loader))

X = X.to(device)
y = y.to(device)

with torch.no_grad():
    yHat,_ = pretrained_net_e(X)
    loss   = loss_fun(yHat,y)

test_loss_pre_e = loss.item()
test_acc_pre_e  = 100*torch.mean((torch.argmax(yHat,axis=1) == y).float()).item()


In [ ]:
# %% Self-inflicted exercise 3
#    Continue ...

# Plotting
phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_loss_pre_e,'s-',color='tab:red',label='Train (pretrained)')
ax[0].plot(dev_loss_pre_e,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[0].plot(len(dev_loss_pre_e)-1,test_loss_pre_e,'*',color='tab:red',markersize=10,label='Test (pretrained)')
ax[0].plot(train_loss_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[0].plot(dev_loss_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[0].plot(len(dev_loss_naive)-1,test_loss_naive,'*',color='tab:blue',markersize=10,label='Test (naïve)')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc_pre_e,'s-',color='tab:red',label='Train (pretrained)')
ax[1].plot(dev_acc_pre_e,'o--',color='tab:red',label='Dev (pretrained)',alpha=0.75)
ax[1].plot(len(dev_acc_pre_e)-1,test_acc_pre_e,'*',color='tab:red',markersize=10,label='Test (pretrained)')
ax[1].plot(train_acc_naive,'s-',color='tab:blue',label='Train (naïve)')
ax[1].plot(dev_acc_naive,'o--',color='tab:blue',label='Dev (naïve)',alpha=0.75)
ax[1].plot(len(dev_acc_naive)-1,test_acc_naive,'*',color='tab:blue',markersize=10,label='Test (naïve)')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final test [naïve, pretrained] accuracy: [{test_acc_naive:.2f}%, {test_acc_pre_e:.2f}%]')
ax[1].legend()

plt.savefig('figure32_pretrain_with_autoencoders_extra3.png')
plt.show()
files.download('figure32_pretrain_with_autoencoders_extra3.png')
